# Week 1R — ProsQA salvage rerun — latent-agent-cost

ProsQA-only rerun of the Week-1R explicit-CoT baseline (Qwen2.5-1.5B-Instruct, ProsQA, 500 tasks x 3 seeds), under the same corrected Week-1R protocol as the combined run.

**Why this notebook exists:** the combined Week-1R run (Version 3 of `week1-baselines.ipynb`, GSM8K + ProsQA in one commit) was killed at Kaggle's ~12 h session limit during the ProsQA cell, after GSM8K had finished. The GSM8K WEEK1R results were already retrieved from that run, so this notebook reruns **only ProsQA** — a single dataset, so the run fits inside one session. Expected duration ~7–8 h, based on the pace of the combined v3 run.

**Before running:** notebook Settings -> **Accelerator = GPU** (T4 or P100) and **Internet = ON** (needed for the git clone, the HF dataset downloads, and the model download).

Run all cells top to bottom — or better, use **Save Version -> Save & Run All (Commit)** so the run executes headless and the outputs persist (see `RUNBOOK.md` in this folder).

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU visible. Set Accelerator = GPU in the notebook settings and re-run.")
print("device:", torch.cuda.get_device_name(0))

Clone the project (fresh every run) and install its dependencies. `requirements.txt` already includes `tokenecon==0.2.0`; the one local pin filtered out is `torch` — Kaggle's preinstalled CUDA build must be kept, never replaced by the local CPU pin.

In [ ]:
!rm -rf latent-agent-cost
!git clone https://github.com/karmendra8386/latent-agent-cost
%cd latent-agent-cost
!grep -v '^torch' requirements.txt > /tmp/req-kaggle.txt
!pip install -q -r /tmp/req-kaggle.txt
!python -c "import torch; print('after install - torch:', torch.__version__, '| cuda:', torch.cuda.is_available())"

ProsQA baseline — 500 tasks x seeds {0, 1, 2}, label `WEEK1R` (the corrected-protocol rerun). Attempts are appended and flushed to `results/week1_prosqa.attempts.jsonl` after every attempt, so a session cut-off keeps completed attempts (the summary tables are written at the end).

In [ ]:
!python scripts/run_baseline.py --dataset prosqa --n 500 --seeds 0 1 2 --model Qwen/Qwen2.5-1.5B-Instruct --device cuda --dtype float16 --out results/week1_prosqa --label WEEK1R

Show the resulting five-metric table.

In [ ]:
!ls -la results/
!cat results/week1_prosqa.md